# 03 Amazon Sentiment Analysis

## Notebook Contract

### Purpose

This notebook performs sentiment analysis on Amazon fashion review data and produces category-aspect sentiment features for recommender system integration.

### Main Input Tables

* `fashion_cip.silver.amazon_aspect_tagged`
* `fashion_cip.silver.amazon_sentiment_input`

### Main Output Tables

* `fashion_cip.silver.amazon_distilbert_aspect_results`
* `fashion_cip.gold.category_aspect_sentiment`
* `fashion_cip.silver.category_aspect_sentiment_quality_check`
* `fashion_cip.silver.category_aspect_sentiment_schema`
* `fashion_cip.silver.category_aspect_top_positive`
* `fashion_cip.silver.category_aspect_top_negative`
* `fashion_cip.silver.category_sentiment_summary`
* `fashion_cip.silver.category_aspect_sentiment_limitations`
* `fashion_cip.silver.sentiment_output_checklist`

### Final Deliverable

The main deliverable from this notebook is:

`fashion_cip.gold.category_aspect_sentiment`

This Gold table contains span-level DistilBERT sentiment scores aggregated by product category and review aspect. It is intended to be used as a sentiment feature input for recommender integration and business reporting.


In [0]:
%pip install vaderSentiment transformers torch -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------
# Step 1: Basic setup
# ---------------------------------------------------------
# Define project catalog, schemas, and raw data path.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"

BRONZE_SCHEMA = f"{CATALOG}.bronze"   # Raw data layer
SILVER_SCHEMA = f"{CATALOG}.silver"   # Cleaned data layer
GOLD_SCHEMA = f"{CATALOG}.gold"       # Final output layer

RAW_PATH = "/Volumes/fashion_cip/bronze/raw"

# Create schemas if needed.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

# Confirm setup.
print("Setup complete")
print("Bronze schema:", BRONZE_SCHEMA)
print("Silver schema:", SILVER_SCHEMA)
print("Gold schema:", GOLD_SCHEMA)
print("Raw path:", RAW_PATH)

Setup complete
Bronze schema: fashion_cip.bronze
Silver schema: fashion_cip.silver
Gold schema: fashion_cip.gold
Raw path: /Volumes/fashion_cip/bronze/raw


In [0]:
# ---------------------------------------------------------
# Step 2: Check raw files
# ---------------------------------------------------------
# List files available in the raw data path.

display(dbutils.fs.ls(RAW_PATH))

path,name,size,modificationTime
dbfs:/Volumes/fashion_cip/bronze/raw/amazon-fashion-user-reviews-dataset.csv,amazon-fashion-user-reviews-dataset.csv,259596390,1781139737000
dbfs:/Volumes/fashion_cip/bronze/raw/articles.csv,articles.csv,36127865,1781139726000
dbfs:/Volumes/fashion_cip/bronze/raw/customers.csv,customers.csv,207135859,1781139737000
dbfs:/Volumes/fashion_cip/bronze/raw/transactions_train.csv,transactions_train.csv,3488002253,1781139787000


In [0]:
# ---------------------------------------------------------
# Step 3: Load Amazon reviews
# ---------------------------------------------------------
# Load the Amazon reviews CSV from the raw data path.

amazon_reviews_path = f"{RAW_PATH}/*amazon*fashion*reviews*.csv"

reviews_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(amazon_reviews_path)
)

# Confirm loading.
print("Amazon reviews loaded")
print("Raw row count:", reviews_raw.count())

display(reviews_raw.limit(10))
reviews_raw.printSchema()

Amazon reviews loaded
Raw row count: 867310


rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchases,target
1.0,It say 5 pair when purchasing but only get 2 rip off,I was looking for 5 pair and only received 2 pair tho I paid for 5 pair and the two pair was late I'm a prime member and feel like I was cheated if my math serve me correct I'm still short 3 pair of panties that were listed as prime,[],B07QFTMTLP,B07QFTMTLP,AHASEZ65RESN57BMGRV6QBM5DTIA,1565088068852,0,True,-1
1.0,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after the first wash and are SO uncomfortable. Buy literally anything else!,[],B0764KKDN1,B0764KKDN1,AE3AMA3QSOHFKV46JJAHTHMMIR6A,1622416429592,0,True,-1
1.0,Small,Retuned is too small for me,[],B07J1WHVCP,B07J1WHVCP,AH4CFWQE2HTC5BSWIEF3LVLUFK6A,1565284666220,0,True,-1
1.0,Pre-Used When Received,This product came with the sleeves turned inside out with lots of stray white hairs on it. It was definitely used before I received it. This will be returned.,[],B0773JWP64,B0773JWP64,AFEKQFJWST6MVTKEJBQKUUBTWK7A,1581963636172,0,False,-1
1.0,Worn once and several places at seams have come apart leaving holes,"Worn once and several places at seams have come apart leaving holes. Shoulder , thigh and crotch areas",[],B099NST9RX,B08JGNS1NK,AGU2FPKN6ARXUSSGBT6WTVLZKJSQ,1640895438476,0,True,-1
1.0,Not true to size or maybe I received a defective one,I ordered a XXL and the shirt I received looked like a small I couldn't get one arm in,[],B072DZ1LDL,B072DZ1LDL,AEJZUZUPFWXOX5G5MA5DIGSZGMYQ,1506874847771,0,True,-1
1.0,One Star,Can't wear it. Way too small.,[],B01HD0ZEC8,B01HD0ZEC8,AHRDWBVOHQNJWSAP4S5JUJ7UAERA,1516064297693,0,True,-1
1.0,Poor quality,Extremely poor qualityAnd extremely poor packaging,[],B00VVT0SYW,B00VVT0SYW,AGLO57O5S5G4SQ67TD4A2ENIZPBA,1545081645271,0,True,-1
1.0,item seems damaged or not made correctly; missing an eye,item seems damaged and/or not made correctly; alligator missing an eye,[],B09DDH9YD5,B09DDH9YD5,AEHVMRUIP33QW4AEMPL4WS5VPAMQ,1668037718632,0,True,-1
1.0,Not for me...,These gloves are not heavy-duty anything. Description is misleading. Sending them back asap.,[],B072QFWQ4T,B072QFWQ4T,AEMDABFWKL7TN5XNWSV5W3CWDANA,1549228061730,1,True,-1


root
 |-- rating: string (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- helpful_vote: string (nullable = true)
 |-- verified_purchases: string (nullable = true)
 |-- target: string (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 4: Check column names
# ---------------------------------------------------------
# Display all columns in the raw reviews dataset.

for column in reviews_raw.columns:
    print(column)

rating
title
text
images
asin
parent_asin
user_id
timestamp
helpful_vote
verified_purchases
target


In [0]:
# ---------------------------------------------------------
# Step 5: Check rating and target values
# ---------------------------------------------------------
# Review rating and target value distribution before cleaning.

rating_check = (
    reviews_raw
    .withColumn("rating_num", F.expr("try_cast(rating as double)"))
    .groupBy("rating_num")
    .count()
    .orderBy("rating_num")
)

target_check = (
    reviews_raw
    .withColumn("target_num", F.expr("try_cast(target as int)"))
    .groupBy("target_num")
    .count()
    .orderBy("target_num")
)

print("Rating distribution:")
display(rating_check)

print("Target distribution:")
display(target_check)

Rating distribution:


rating_num,count
1.0,173462
2.0,173462
3.0,173462
4.0,173462
5.0,173462


Target distribution:


target_num,count
-1,346924
0,173462
1,346924


In [0]:
# ---------------------------------------------------------
# Step 6: Clean review data
# ---------------------------------------------------------
# Standardize data types and create analysis-ready columns.

reviews_clean = (
    reviews_raw
    .withColumn("rating_num", F.expr("try_cast(rating as double)"))
    .withColumn("target_num", F.expr("try_cast(target as int)"))
    .withColumn("helpful_vote_num", F.expr("try_cast(helpful_vote as int)"))
    .withColumn("timestamp_ms", F.expr("try_cast(timestamp as long)"))
    .withColumn("review_text", F.trim(F.col("text").cast("string")))
    .withColumn("review_title", F.trim(F.col("title").cast("string")))
    .withColumn("review_timestamp", F.to_timestamp(F.from_unixtime((F.col("timestamp_ms") / 1000).cast("long"))))
    .withColumn("review_date", F.to_date("review_timestamp"))
    .filter(F.col("review_text").isNotNull())
    .filter(F.length(F.col("review_text")) > 0)
    .filter(F.col("rating_num").isin([1.0, 2.0, 3.0, 4.0, 5.0]))
    .filter(F.col("target_num").isin([-1, 0, 1]))
)

# Create a unique review ID.
reviews_clean = reviews_clean.withColumn(
    "review_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("user_id").cast("string"), F.lit("")),
            F.coalesce(F.col("asin").cast("string"), F.lit("")),
            F.coalesce(F.col("parent_asin").cast("string"), F.lit("")),
            F.coalesce(F.col("timestamp_ms").cast("string"), F.lit("")),
            F.coalesce(F.col("review_text").cast("string"), F.lit(""))
        ),
        256
    )
)

# Select final columns.
reviews_clean = reviews_clean.select(
    "review_id",
    F.col("user_id").cast("string").alias("user_id"),
    F.col("asin").cast("string").alias("asin"),
    F.col("parent_asin").cast("string").alias("parent_asin"),
    F.col("review_title").alias("review_title"),
    F.col("review_text").alias("review_text"),
    F.col("rating_num").alias("rating"),
    F.col("target_num").alias("target"),
    F.col("helpful_vote_num").alias("helpful_vote"),
    F.col("verified_purchases").cast("string").alias("verified_purchase"),
    F.col("review_timestamp"),
    F.col("review_date")
)

# Confirm cleaned data.
print("Cleaned row count:", reviews_clean.count())

display(reviews_clean.limit(10))
reviews_clean.printSchema()

Cleaned row count: 866575


review_id,user_id,asin,parent_asin,review_title,review_text,rating,target,helpful_vote,verified_purchase,review_timestamp,review_date
39827f39b2998407ff4650554bd146c8ae7ffda179245c34a6c2a83cb61e1f20,AHASEZ65RESN57BMGRV6QBM5DTIA,B07QFTMTLP,B07QFTMTLP,It say 5 pair when purchasing but only get 2 rip off,I was looking for 5 pair and only received 2 pair tho I paid for 5 pair and the two pair was late I'm a prime member and feel like I was cheated if my math serve me correct I'm still short 3 pair of panties that were listed as prime,1.0,-1,0,True,2019-08-06T10:41:08.000Z,2019-08-06
49b27d7e84e743204df8a8c20afd32b1a0850657e4c14b274d9b138c2e099b12,AE3AMA3QSOHFKV46JJAHTHMMIR6A,B0764KKDN1,B0764KKDN1,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after the first wash and are SO uncomfortable. Buy literally anything else!,1.0,-1,0,True,2021-05-30T23:13:49.000Z,2021-05-30
a966ac5657a25a4a68ac79671bc8b0715651a0697a4e2cab677187fe62156157,AH4CFWQE2HTC5BSWIEF3LVLUFK6A,B07J1WHVCP,B07J1WHVCP,Small,Retuned is too small for me,1.0,-1,0,True,2019-08-08T17:17:46.000Z,2019-08-08
fa62345f562b92b281b7588222c00f958c8ab83c64cea7ed5e354747e36f9ebb,AFEKQFJWST6MVTKEJBQKUUBTWK7A,B0773JWP64,B0773JWP64,Pre-Used When Received,This product came with the sleeves turned inside out with lots of stray white hairs on it. It was definitely used before I received it. This will be returned.,1.0,-1,0,False,2020-02-17T18:20:36.000Z,2020-02-17
64fc9326ba50d2743e9d80e285601f0408de81a92d9ac2d9a8de4b45106ef72d,AGU2FPKN6ARXUSSGBT6WTVLZKJSQ,B099NST9RX,B08JGNS1NK,Worn once and several places at seams have come apart leaving holes,"Worn once and several places at seams have come apart leaving holes. Shoulder , thigh and crotch areas",1.0,-1,0,True,2021-12-30T20:17:18.000Z,2021-12-30
4aa60f8acd8f4185b82152900e65ae66cc22c4c15f8b43e7c63f913317b262c0,AEJZUZUPFWXOX5G5MA5DIGSZGMYQ,B072DZ1LDL,B072DZ1LDL,Not true to size or maybe I received a defective one,I ordered a XXL and the shirt I received looked like a small I couldn't get one arm in,1.0,-1,0,True,2017-10-01T16:20:47.000Z,2017-10-01
1ffca36ece1ec03afaf5335ec3e9ec39c133296f0588965db72700fabb636f54,AHRDWBVOHQNJWSAP4S5JUJ7UAERA,B01HD0ZEC8,B01HD0ZEC8,One Star,Can't wear it. Way too small.,1.0,-1,0,True,2018-01-16T00:58:17.000Z,2018-01-16
c7b2e9fdf178d1fc82b0977e6faf8e008575107108101b1ea263adf83c38888b,AGLO57O5S5G4SQ67TD4A2ENIZPBA,B00VVT0SYW,B00VVT0SYW,Poor quality,Extremely poor qualityAnd extremely poor packaging,1.0,-1,0,True,2018-12-17T21:20:45.000Z,2018-12-17
835d412ff7391c0c6507189fbbca9b7a9d36bc158400319b1bb0a42e8f3d518d,AEHVMRUIP33QW4AEMPL4WS5VPAMQ,B09DDH9YD5,B09DDH9YD5,item seems damaged or not made correctly; missing an eye,item seems damaged and/or not made correctly; alligator missing an eye,1.0,-1,0,True,2022-11-09T23:48:38.000Z,2022-11-09
66e59fff05f82578cad73172d45c75d47fed3226011a85f509ff90d394d66c9c,AEMDABFWKL7TN5XNWSV5W3CWDANA,B072QFWQ4T,B072QFWQ4T,Not for me...,These gloves are not heavy-duty anything. Description is misleading. Sending them back asap.,1.0,-1,1,True,2019-02-03T21:07:41.000Z,2019-02-03


root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- target: integer (nullable = true)
 |-- helpful_vote: integer (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 7: Save cleaned reviews
# ---------------------------------------------------------
# Save the cleaned review dataset for sentiment analysis.

reviews_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_reviews_clean"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_reviews_clean")

Saved table: fashion_cip.silver.amazon_reviews_clean


In [0]:
# ---------------------------------------------------------
# Step 8: Verify saved table
# ---------------------------------------------------------
# Read the saved Silver table and check the output.

reviews_df = spark.table(f"{SILVER_SCHEMA}.amazon_reviews_clean")

print("Row count:", reviews_df.count())
print("Column count:", len(reviews_df.columns))

display(reviews_df.limit(10))
reviews_df.printSchema()

Row count: 866575
Column count: 12


review_id,user_id,asin,parent_asin,review_title,review_text,rating,target,helpful_vote,verified_purchase,review_timestamp,review_date
39827f39b2998407ff4650554bd146c8ae7ffda179245c34a6c2a83cb61e1f20,AHASEZ65RESN57BMGRV6QBM5DTIA,B07QFTMTLP,B07QFTMTLP,It say 5 pair when purchasing but only get 2 rip off,I was looking for 5 pair and only received 2 pair tho I paid for 5 pair and the two pair was late I'm a prime member and feel like I was cheated if my math serve me correct I'm still short 3 pair of panties that were listed as prime,1.0,-1,0,True,2019-08-06T10:41:08.000Z,2019-08-06
49b27d7e84e743204df8a8c20afd32b1a0850657e4c14b274d9b138c2e099b12,AE3AMA3QSOHFKV46JJAHTHMMIR6A,B0764KKDN1,B0764KKDN1,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after the first wash and are SO uncomfortable. Buy literally anything else!,1.0,-1,0,True,2021-05-30T23:13:49.000Z,2021-05-30
a966ac5657a25a4a68ac79671bc8b0715651a0697a4e2cab677187fe62156157,AH4CFWQE2HTC5BSWIEF3LVLUFK6A,B07J1WHVCP,B07J1WHVCP,Small,Retuned is too small for me,1.0,-1,0,True,2019-08-08T17:17:46.000Z,2019-08-08
fa62345f562b92b281b7588222c00f958c8ab83c64cea7ed5e354747e36f9ebb,AFEKQFJWST6MVTKEJBQKUUBTWK7A,B0773JWP64,B0773JWP64,Pre-Used When Received,This product came with the sleeves turned inside out with lots of stray white hairs on it. It was definitely used before I received it. This will be returned.,1.0,-1,0,False,2020-02-17T18:20:36.000Z,2020-02-17
64fc9326ba50d2743e9d80e285601f0408de81a92d9ac2d9a8de4b45106ef72d,AGU2FPKN6ARXUSSGBT6WTVLZKJSQ,B099NST9RX,B08JGNS1NK,Worn once and several places at seams have come apart leaving holes,"Worn once and several places at seams have come apart leaving holes. Shoulder , thigh and crotch areas",1.0,-1,0,True,2021-12-30T20:17:18.000Z,2021-12-30
4aa60f8acd8f4185b82152900e65ae66cc22c4c15f8b43e7c63f913317b262c0,AEJZUZUPFWXOX5G5MA5DIGSZGMYQ,B072DZ1LDL,B072DZ1LDL,Not true to size or maybe I received a defective one,I ordered a XXL and the shirt I received looked like a small I couldn't get one arm in,1.0,-1,0,True,2017-10-01T16:20:47.000Z,2017-10-01
1ffca36ece1ec03afaf5335ec3e9ec39c133296f0588965db72700fabb636f54,AHRDWBVOHQNJWSAP4S5JUJ7UAERA,B01HD0ZEC8,B01HD0ZEC8,One Star,Can't wear it. Way too small.,1.0,-1,0,True,2018-01-16T00:58:17.000Z,2018-01-16
c7b2e9fdf178d1fc82b0977e6faf8e008575107108101b1ea263adf83c38888b,AGLO57O5S5G4SQ67TD4A2ENIZPBA,B00VVT0SYW,B00VVT0SYW,Poor quality,Extremely poor qualityAnd extremely poor packaging,1.0,-1,0,True,2018-12-17T21:20:45.000Z,2018-12-17
835d412ff7391c0c6507189fbbca9b7a9d36bc158400319b1bb0a42e8f3d518d,AEHVMRUIP33QW4AEMPL4WS5VPAMQ,B09DDH9YD5,B09DDH9YD5,item seems damaged or not made correctly; missing an eye,item seems damaged and/or not made correctly; alligator missing an eye,1.0,-1,0,True,2022-11-09T23:48:38.000Z,2022-11-09
66e59fff05f82578cad73172d45c75d47fed3226011a85f509ff90d394d66c9c,AEMDABFWKL7TN5XNWSV5W3CWDANA,B072QFWQ4T,B072QFWQ4T,Not for me...,These gloves are not heavy-duty anything. Description is misleading. Sending them back asap.,1.0,-1,1,True,2019-02-03T21:07:41.000Z,2019-02-03


root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- target: integer (nullable = true)
 |-- helpful_vote: integer (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 9: Check rating and target distribution
# ---------------------------------------------------------
# Summarize ratings and sentiment labels.

total_reviews = reviews_df.count()

rating_distribution = (
    reviews_df
    .groupBy("rating")
    .count()
    .withColumn("percentage", F.round((F.col("count") / total_reviews) * 100, 2))
    .orderBy("rating")
)

target_distribution = (
    reviews_df
    .groupBy("target")
    .count()
    .withColumn("percentage", F.round((F.col("count") / total_reviews) * 100, 2))
    .orderBy("target")
)

print("Rating distribution:")
display(rating_distribution)

print("Target distribution:")
display(target_distribution)

Rating distribution:


rating,count,percentage
1.0,173271,19.99
2.0,173383,20.01
3.0,173370,20.01
4.0,173322,20.0
5.0,173229,19.99


Target distribution:


target,count,percentage
-1,346654,40.0
0,173370,20.01
1,346551,39.99


In [0]:
# ---------------------------------------------------------
# Step 10: Create EDA summary
# ---------------------------------------------------------
# Generate basic dataset-level summary statistics.

eda_summary = reviews_df.select(
    F.count("*").alias("total_reviews"),
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("asin").alias("unique_products"),
    F.countDistinct("parent_asin").alias("unique_parent_products"),
    F.round(F.avg("rating"), 2).alias("average_rating"),
    F.round(F.avg(F.length("review_text")), 2).alias("average_review_length"),
    F.min(F.length("review_text")).alias("min_review_length"),
    F.max(F.length("review_text")).alias("max_review_length")
)

display(eda_summary)

total_reviews,unique_users,unique_products,unique_parent_products,average_rating,average_review_length,min_review_length,max_review_length
866575,782264,447182,424527,3.0,158.43,1,14382


In [0]:
# ---------------------------------------------------------
# Step 11: Create stratified sample
# ---------------------------------------------------------
# Create a 50K sample while preserving target distribution.

from pyspark.sql.window import Window

sample_size = 50000
total_reviews = reviews_df.count()

# Calculate sample size for each target class.
target_counts = reviews_df.groupBy("target").count().collect()

sample_sizes = {
    row["target"]: round((row["count"] / total_reviews) * sample_size)
    for row in target_counts
}

print("Target sample sizes:", sample_sizes)

# Add row numbers within each target class.
window_spec = Window.partitionBy("target").orderBy(F.rand(seed=42))

sample_df = (
    reviews_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(
        ((F.col("target") == -1) & (F.col("row_num") <= sample_sizes[-1])) |
        ((F.col("target") == 0) & (F.col("row_num") <= sample_sizes[0])) |
        ((F.col("target") == 1) & (F.col("row_num") <= sample_sizes[1]))
    )
    .drop("row_num")
)

print("Sample row count:", sample_df.count())

display(
    sample_df
    .groupBy("target")
    .count()
    .orderBy("target")
)

Target sample sizes: {-1: 20001, 0: 10003, 1: 19995}
Sample row count: 49999


target,count
-1,20001
0,10003
1,19995


In [0]:
# ---------------------------------------------------------
# Step 12: Save stratified sample
# ---------------------------------------------------------
# Save the 50K sample for baseline sentiment analysis.

sample_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_sentiment_sample_50k"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_sentiment_sample_50k")

Saved table: fashion_cip.silver.amazon_sentiment_sample_50k


In [0]:
# ---------------------------------------------------------
# Step 13: Run VADER baseline
# ---------------------------------------------------------
# Apply VADER sentiment scoring to the sampled reviews.

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

sample_df = spark.table(f"{SILVER_SCHEMA}.amazon_sentiment_sample_50k")

# Convert sample to Pandas for VADER processing.
vader_pdf = sample_df.select(
    "review_id",
    "rating",
    "target",
    "review_text"
).toPandas()

# Initialize VADER sentiment analyzer.
analyzer = SentimentIntensityAnalyzer()

# Calculate VADER compound sentiment score.
def get_vader_score(text):
    if pd.isna(text):
        return 0.0
    return analyzer.polarity_scores(str(text))["compound"]

vader_pdf["vader_compound"] = vader_pdf["review_text"].apply(get_vader_score)

# Convert compound score into sentiment label.
def get_vader_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

vader_pdf["vader_label"] = vader_pdf["vader_compound"].apply(get_vader_label)

print("VADER scoring completed")
print("Rows scored:", len(vader_pdf))

display(spark.createDataFrame(vader_pdf).limit(10))

VADER scoring completed
Rows scored: 49999


review_id,rating,target,review_text,vader_compound,vader_label
f837ae871451fd056783ac28c9c8ece685b37dea6e7b191202f7d48c625c87c1,1.0,-1,"Small is huge, fits awkwardly.",0.0,neutral
60c7faa014e03bb2893b34a8cc856a30ad8bfe8cd126239c9b3fcbaf37f35ac7,1.0,-1,I didn't like the fuzzy fabric. The buttons are not cute and it made me feel older. Based on the pictures I thought it looked nice but when it finally came in I realized it's just not my style.,-0.1015,negative
75bcf43a42034f4f55f19065df3b0449b3207da55e9f24eb89cb29ecdde2b1eb,1.0,-1,I returned these. They would not hold the backs on and looked cheaply made.,0.0382,neutral
fb9266453820953871e0bc1c44f8ff98221be24b7d3ce4ca8a0bc8dff10f36ee,1.0,-1,Mala calidad,0.0,neutral
0206d800c63169be250c22517d4a765b94a7a1fd529f01f37c050db2d4f427bb,2.0,-1,The heart fell out of the silver frame the first time I wore it.,0.6369,positive
a2958dd793239e58ab15a78b91edd5a29f614928c34395c9fc2a1e581cd8bd28,1.0,-1,"Does not look like the picture! The top portion is soooooo short, it barely comes underneath my breast. ItÃ¢ÂÂs much longer on the model. Also IÃ¢ÂÂm short, 5Ã¢ÂÂ4 and it hangs several inches off the floor. Pattern does not match to the photo. Stitching connecting the top to bottom is barely holding on.",0.1979,positive
143620c3a0f3334d1e60f65f1d310d2637f1767bf3be772dcbcb37564eb0d7e8,1.0,-1,No ear loop,-0.296,negative
2e53790280cf8d05483e5a891c6017b4222687cb4507dca36e7ca0afb886b983,1.0,-1,"Returned for full refundThe label said 100% cotton, but I donÃ¢ÂÂt really think thatÃ¢ÂÂs what the fabric really is made of. It felt more like a man-made fabric and like rayon. It also required dry cleaning Ã°ÂÂÂÃ°ÂÂÂ»Ã°ÂÂÂÃ°ÂÂÂ»I have serval linen and cotton pants, and itÃ¢ÂÂs definitely not made from any of these types of fabrics. Ã¢ÂÂ¹Ã¯Â¸Â",0.8883,positive
31239e6c2511bb1d1614bba906c44f2edcb70c8cba913c53170ee9f237d5c2f4,1.0,-1,"I feel so embarrassed, given to my sister as a gift,after a one week, the ring became discolored, definitely is NOT! ! 925 sterling silver, I recommended do not buy.very poor quality.",0.5262,positive
f3f35af10503305749700875380ec9fd12670703d9575a2ea4a94c397b206bcd,2.0,-1,"One of the fitting brackets on the bottom row broke off on the first day. They have been pushing my gums down from the grill on my bottom row. The upper fangs are causing my lips to bleed, over all not the best product.",-0.7326,negative


In [0]:
# ---------------------------------------------------------
# Step 14: Validate VADER scores
# ---------------------------------------------------------
# Check whether VADER sentiment aligns with ratings and target labels.

corr_rating = vader_pdf["vader_compound"].corr(vader_pdf["rating"])
corr_target = vader_pdf["vader_compound"].corr(vader_pdf["target"])

print("Correlation with rating:", round(corr_rating, 4))
print("Correlation with target:", round(corr_target, 4))

# Average VADER score by rating.
vader_by_rating = (
    vader_pdf
    .groupby("rating")
    .agg(
        review_count=("review_id", "count"),
        mean_vader_score=("vader_compound", "mean")
    )
    .reset_index()
)

display(spark.createDataFrame(vader_by_rating))

Correlation with rating: 0.5597
Correlation with target: 0.5443


rating,review_count,mean_vader_score
1.0,9978,-0.07866362998596914
2.0,10023,0.11906439189863315
3.0,10003,0.30714262721183644
4.0,10044,0.5752298287534847
5.0,9951,0.6844061400864234


In [0]:
# ---------------------------------------------------------
# Step 15: Save VADER results
# ---------------------------------------------------------
# Save scored reviews for later comparison and reporting.

vader_results_df = spark.createDataFrame(vader_pdf)

vader_results_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_vader_results_50k"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_vader_results_50k")
print("Saved row count:", vader_results_df.count())

Saved table: fashion_cip.silver.amazon_vader_results_50k
Saved row count: 49999


In [0]:
# ---------------------------------------------------------
# Step 16: Verify VADER results
# ---------------------------------------------------------
# Check saved VADER output table.

vader_results_df = spark.table(f"{SILVER_SCHEMA}.amazon_vader_results_50k")

print("Row count:", vader_results_df.count())
print("Column count:", len(vader_results_df.columns))

display(vader_results_df.limit(10))
vader_results_df.printSchema()

Row count: 49999
Column count: 6


review_id,rating,target,review_text,vader_compound,vader_label
b6ac6216cf245903f0ef7c4f971f36b1ae9ea5fa9a96770ed9d6f4eae86d29f8,4.0,1,good as expected,0.4404,positive
7752c9ac519e9abdfebf047f557864c8c1cc1baee4785fccbb708c6e338359ed,4.0,1,The bird itself is huge. The stickers are no larger than previous draft boards weÃ¢ÂÂve used. I would purchase again next season,0.0258,neutral
9cbc103797b3dbc0c2dbae594b7e4c705f87caca8fb3651a8acf09b095fc17f8,4.0,1,Very cute but it is a bit short for me (not long enough for me to comfortably sit down).,0.7047,positive
174e7bd5bd67949fd844c896f859ad4d537d6953e762dc4d49f3a90572de8f10,5.0,1,love it. well made,0.743,positive
83cdaab7406db9d1ed880dad3bad29e8884da9f3f267946691df9f140598850f,4.0,1,"Rec'd quickly, quality is OK. However, they are quite wrinkled after coming out of the dryer, which I wouldn't expect. Probably would not purchase again.",0.0,neutral
c5cd34b8101163d0a21b3cd6bc83727f0e51ddf578ec954253b6e90073cd5bea,4.0,1,"I typically wear an XL, but I bought in a XXL to be safe and that was the right move. It is so cute. Just be warned though, for some reason looking at the picture I thought the knees were like ripped, but it is actually two pieces - one above the knee and one below the knee- that are sown together on the sides. It was surprising but still cute.",0.9545,positive
9be7337ab33ff96b50c8144bef601be250c708412369ddbadc0ed45e3204b79b,5.0,1,"These are my favorite glasses. I have them in all available colors, and the only thing that could improve them would be MORE COLORS!! Their lenses are some of the best and never have weird wavy areas making vision go wonky, and I get compliments on them all of the time. I lost my other pair this of this color in the house, so I bought a second pair because they're my favorite. Lavender, light olive green, and light turquoise would also make me very happy if you're listening SOOLALA!!",0.969,positive
c6cb113f1e1b5e3395ff01c6a386ca3a64ea34e753f270671e6235448a6d278a,4.0,1,"This Moyabo womenÃ¢ÂÂs sleeveless V-neck dress is cute! The top runs a little small, so be warned. The top of the dress is a black, solid-colored tank top with a scoop-shaped, crisscross neckline, which makes it casual and cute. The skirt is flared and a colorful floral print. It even has side pockets for your convenience. This is knee length on me. This dress is easy to care for. All you do is machine or hand wash in cold water, then hang dry. It is wrinkle resistant, so it would be great for travel. This dress moves with you and hangs nicely. The dress is 95% cotton and 5% spandex. It is comfortable and pretty. This dress could go casual or dressy, depending upon your accessories.",0.9858,positive
58e6f37cc876d877778d4734afd3253f46391570d1787d615b03e04da0b4cd80,5.0,1,Lost the beads we bought in Nepal...,-0.3182,negative
53d3b0564ab1e4f34cab4f647eb26d7d25ea05ebb9296c2a6940bd9d21b025ef,5.0,1,"Great Seller, Love the item, gift for a friend and they loved it as well!",0.967,positive


root
 |-- review_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- target: integer (nullable = true)
 |-- review_text: string (nullable = true)
 |-- vader_compound: double (nullable = true)
 |-- vader_label: string (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 17: Summarize VADER results
# ---------------------------------------------------------
# Create summary tables for baseline sentiment results.

# Average VADER score by rating.
vader_summary_by_rating = (
    vader_results_df
    .groupBy("rating")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("vader_compound"), 4).alias("mean_vader_score")
    )
    .orderBy("rating")
)

# VADER label distribution.
vader_label_distribution = (
    vader_results_df
    .groupBy("vader_label")
    .agg(F.count("*").alias("review_count"))
    .orderBy("vader_label")
)

print("VADER summary by rating:")
display(vader_summary_by_rating)

print("VADER label distribution:")
display(vader_label_distribution)

VADER summary by rating:


rating,review_count,mean_vader_score
1.0,9978,-0.0787
2.0,10023,0.1191
3.0,10003,0.3071
4.0,10044,0.5752
5.0,9951,0.6844


VADER label distribution:


vader_label,review_count
negative,10405
neutral,7287
positive,32307


In [0]:
# ---------------------------------------------------------
# Step 18: Save VADER summary tables
# ---------------------------------------------------------
# Save baseline sentiment summaries for reporting.

vader_summary_by_rating.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_vader_summary_by_rating"
)

vader_label_distribution.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_vader_label_distribution"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_vader_summary_by_rating")
print("Saved table:", f"{SILVER_SCHEMA}.amazon_vader_label_distribution")

Saved table: fashion_cip.silver.amazon_vader_summary_by_rating
Saved table: fashion_cip.silver.amazon_vader_label_distribution


In [0]:
# ---------------------------------------------------------
# Step 19: Log VADER baseline to MLflow
# ---------------------------------------------------------
# Track baseline sentiment model results.

import mlflow

# Load saved VADER results.
vader_results_df = spark.table(f"{SILVER_SCHEMA}.amazon_vader_results_50k")
vader_pdf_log = vader_results_df.select(
    "rating",
    "target",
    "vader_compound"
).toPandas()

# Calculate validation metrics.
corr_rating = vader_pdf_log["vader_compound"].corr(vader_pdf_log["rating"])
corr_target = vader_pdf_log["vader_compound"].corr(vader_pdf_log["target"])
avg_vader_score = vader_pdf_log["vader_compound"].mean()

# Set MLflow experiment.
mlflow.set_experiment("/Shared/fashion_cip_kanishka_sentiment")

with mlflow.start_run(run_name="vader_baseline_50k"):
    mlflow.log_param("model", "VADER")
    mlflow.log_param("sample_size", len(vader_pdf_log))
    mlflow.log_param("input_table", f"{SILVER_SCHEMA}.amazon_sentiment_sample_50k")
    mlflow.log_param("output_table", f"{SILVER_SCHEMA}.amazon_vader_results_50k")
    
    mlflow.log_metric("corr_vader_rating", float(corr_rating))
    mlflow.log_metric("corr_vader_target", float(corr_target))
    mlflow.log_metric("avg_vader_score", float(avg_vader_score))

print("MLflow logging completed")
print("Correlation with rating:", round(corr_rating, 4))
print("Correlation with target:", round(corr_target, 4))

MLflow logging completed
Correlation with rating: 0.5597
Correlation with target: 0.5443


In [0]:
# ---------------------------------------------------------
# Step 20: Test DistilBERT on sample
# ---------------------------------------------------------
# Run DistilBERT sentiment scoring on a small sample.

from transformers import pipeline
import time
import pandas as pd

# Load the saved 50K sample.
sample_df = spark.table(f"{SILVER_SCHEMA}.amazon_sentiment_sample_50k")

# Select 1,000 reviews for initial testing.
distilbert_test_pdf = (
    sample_df
    .select("review_id", "rating", "target", "review_text")
    .orderBy(F.rand(seed=42))
    .limit(1000)
    .toPandas()
)

print("Test sample size:", len(distilbert_test_pdf))

# Load pre-trained DistilBERT sentiment model.
distilbert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True
)

# Run sentiment prediction and track time.
texts = distilbert_test_pdf["review_text"].astype(str).tolist()

start_time = time.time()

distilbert_outputs = distilbert_model(
    texts,
    batch_size=32,
    truncation=True,
    max_length=512
)

end_time = time.time()

# Store prediction results.
distilbert_test_pdf["distilbert_label"] = [item["label"] for item in distilbert_outputs]
distilbert_test_pdf["distilbert_score"] = [item["score"] for item in distilbert_outputs]

runtime_seconds = end_time - start_time

print("DistilBERT scoring completed")
print("Runtime seconds:", round(runtime_seconds, 2))
print("Average seconds per review:", round(runtime_seconds / len(distilbert_test_pdf), 4))

display(spark.createDataFrame(distilbert_test_pdf).limit(10))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-fa86747f-0134-4d72-b117-b7f4dd2eb09f/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


Test sample size: 1000


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT scoring completed
Runtime seconds: 90.29
Average seconds per review: 0.0903


review_id,rating,target,review_text,distilbert_label,distilbert_score
5d8ad44003a6be75c5e125699273e1c86dd691b06fcc67ab504574038bef5ac1,4.0,1,"For reference I am 5'6"". The blouse fits great in the shoulders, bust, and waist. However the sleeves are very short, reaching about four inches above my wrist bones. It is fairly short as well, not reaching my hip bones, but as it is worn tucked in, that's fine.I usually find shirt sleeves a bit too long so it is a very weird problem for me too haveI received a lot of compliments on the blouse! It is pretty and priced well.",POSITIVE,0.9934605360031128
53381116a0438544f05c950c5ac949eca9ecd9cc7cb467be968eb1f25e08aed0,3.0,0,Love the color and this styleQuality so far seems to be fair for the costOnly dislike the adjustment part is very stiff but maybe with time will break in and adjust easier,POSITIVE,0.9843632578849792
0206d800c63169be250c22517d4a765b94a7a1fd529f01f37c050db2d4f427bb,2.0,-1,The heart fell out of the silver frame the first time I wore it.,NEGATIVE,0.9975447058677673
c2c4dbca33ffde6c29415d26f769a354590d2f2387debe3ebdb15bd6cd603826,2.0,-1,"I am in love with this dress. I ordered the xxl because of reviews, which turns out to be a little too big. Luckily itÃ¢ÂÂs a true wrap dress and is an easy fix! IÃ¢ÂÂll be wearing this for my publishing debut and my graduation; so pleased.Edit: I bought another dress in a different color and they sent THE WRONG DRESS. So disappointed photo is of the second dress they sent me",NEGATIVE,0.9838870167732239
08fc0f030fe54070dc59dff61af1345de745e2d3e3519ff7e3c776edc0d77c7e,4.0,1,Snugs right on shouler,NEGATIVE,0.9818923473358154
fc9a0b552610119b505787f6fc7c03aeb2c92ed3a498285fa91dafa97e658efd,3.0,0,"The shirts are nice and light and fit well under the arms. Unfortunately they are a bit short for me, as I am 6'5 223 lbs. I am rocking a smaller dad bod, but most of the places they are tight is in the chest, and shoulder areas. They hang a little loose off my midsection, but tighten up quite a bit as soon as it hits my lower chest. They sent me a 3XL which I was luck they dis, as I thi k a 2X may have been way too small.",NEGATIVE,0.9252110123634338
1877f0fda366ab3770519a4020dd45fae906d6d52a1b650e6b4d8eb4c3415ed3,2.0,-1,I ordered an extra large and it was more like a small. Very pretty sure but I definitely will be returning it,POSITIVE,0.9976145029067993
915730d627c6e707a45ec19ed9403b00a6668d0e959f03cced3ac0ba4b6b3ebf,2.0,-1,"(Caveat: I have changed my review downward after wearing these panties!) This particular brand is cut badly, and falls below the belly button, and once on, they roll down and become quite uncomfortable. I do not recommend this panty any longer! I love this type of panty so much I have a dozen or more pairs! They are great for travel and the hidden pocket is a great place to stash your passport or money (I usually put them into a plastic bag with seals first). I got them for my sister and she also loves them. The secret is to buy a size larger than you think you wear for good fit, and since they are quite elastic, you still get great support. In this particular brand I would buy at least two to three sizes larger, and hope the cut suits your body. They do not work for me, as they roll down and tuck in below my bikini line surgical scar, where my belly flaps over - exactly the part of my body I wanted the coverage for!",POSITIVE,0.7505422234535217
e5becf9cda80cf28dea4e53cf57eac79321e8fcbae9afd9338d5169480b1725d,2.0,-1,the bars bend on the first time I wore them,NEGATIVE,0.8504220247268677
85ee97f4e9179705ba7ca08bafa83de073c7a11fd71d9de3c67110844d7eb23e,4.0,1,"Perfect fit. It starts pilling a bit after the first or second machine wash, but IÃ¢ÂÂve washed it many times and itÃ¢ÂÂs still wearable. Soft fabric, would recommend",NEGATIVE,0.9587413668632507


In [0]:
# ---------------------------------------------------------
# Step 21: Validate DistilBERT test results
# ---------------------------------------------------------
# Check whether DistilBERT sentiment aligns with ratings and target labels.

import numpy as np

# Convert DistilBERT labels into signed sentiment scores.
distilbert_test_pdf["distilbert_sentiment_score"] = np.where(
    distilbert_test_pdf["distilbert_label"] == "POSITIVE",
    distilbert_test_pdf["distilbert_score"],
    -distilbert_test_pdf["distilbert_score"]
)

# Calculate correlation with rating and target.
distilbert_corr_rating = distilbert_test_pdf["distilbert_sentiment_score"].corr(
    distilbert_test_pdf["rating"]
)

distilbert_corr_target = distilbert_test_pdf["distilbert_sentiment_score"].corr(
    distilbert_test_pdf["target"]
)

print("Correlation with rating:", round(distilbert_corr_rating, 4))
print("Correlation with target:", round(distilbert_corr_target, 4))

# Average DistilBERT score by rating.
distilbert_summary_by_rating = (
    distilbert_test_pdf
    .groupby("rating")
    .agg(
        review_count=("review_id", "count"),
        mean_distilbert_score=("distilbert_sentiment_score", "mean")
    )
    .reset_index()
)

display(spark.createDataFrame(distilbert_summary_by_rating))

Correlation with rating: 0.6439
Correlation with target: 0.6277


rating,review_count,mean_distilbert_score
1.0,208,-0.8502774361807567
2.0,208,-0.6354811008159931
3.0,170,-0.3866328958202811
4.0,225,0.3871845356623332
5.0,189,0.8388036710244638


In [0]:
# ---------------------------------------------------------
# Step 22: Save DistilBERT test results
# ---------------------------------------------------------
# Save the 1K DistilBERT test output.

distilbert_test_results_df = spark.createDataFrame(distilbert_test_pdf)

distilbert_test_results_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_distilbert_test_1k"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_distilbert_test_1k")
print("Saved row count:", distilbert_test_results_df.count())

Saved table: fashion_cip.silver.amazon_distilbert_test_1k
Saved row count: 1000


In [0]:
# ---------------------------------------------------------
# Step 23: Log DistilBERT test to MLflow
# ---------------------------------------------------------
# Track DistilBERT test results and runtime.

import mlflow

# Use the saved DistilBERT test output.
distilbert_log_df = spark.table(f"{SILVER_SCHEMA}.amazon_distilbert_test_1k")

distilbert_log_pdf = distilbert_log_df.select(
    "rating",
    "target",
    "distilbert_sentiment_score"
).toPandas()

# Recalculate validation metrics.
distilbert_corr_rating = distilbert_log_pdf["distilbert_sentiment_score"].corr(
    distilbert_log_pdf["rating"]
)

distilbert_corr_target = distilbert_log_pdf["distilbert_sentiment_score"].corr(
    distilbert_log_pdf["target"]
)

avg_distilbert_score = distilbert_log_pdf["distilbert_sentiment_score"].mean()

# Log experiment details.
mlflow.set_experiment("/Shared/fashion_cip_kanishka_sentiment")

with mlflow.start_run(run_name="distilbert_test_1k"):
    mlflow.log_param("model", "distilbert-base-uncased-finetuned-sst-2-english")
    mlflow.log_param("sample_size", len(distilbert_log_pdf))
    mlflow.log_param("input_table", f"{SILVER_SCHEMA}.amazon_sentiment_sample_50k")
    mlflow.log_param("output_table", f"{SILVER_SCHEMA}.amazon_distilbert_test_1k")
    
    mlflow.log_metric("corr_distilbert_rating", float(distilbert_corr_rating))
    mlflow.log_metric("corr_distilbert_target", float(distilbert_corr_target))
    mlflow.log_metric("avg_distilbert_score", float(avg_distilbert_score))
    
    if "runtime_seconds" in globals():
        mlflow.log_metric("runtime_seconds", float(runtime_seconds))
        mlflow.log_metric("seconds_per_review", float(runtime_seconds / len(distilbert_log_pdf)))

print("MLflow logging completed")
print("Correlation with rating:", round(distilbert_corr_rating, 4))
print("Correlation with target:", round(distilbert_corr_target, 4))

MLflow logging completed
Correlation with rating: 0.6439
Correlation with target: 0.6277


In [0]:
# ---------------------------------------------------------
# Step 24: Compare sentiment models
# ---------------------------------------------------------
# Compare VADER and DistilBERT validation results.

comparison_data = [
    {
        "model": "VADER",
        "sample_size": 50000,
        "correlation_with_rating": float(0.5597),
        "correlation_with_target": float(0.5443),
        "notes": "Lexicon-based baseline model"
    },
    {
        "model": "DistilBERT",
        "sample_size": 1000,
        "correlation_with_rating": float(round(distilbert_corr_rating, 4)),
        "correlation_with_target": float(round(distilbert_corr_target, 4)),
        "notes": "Transformer-based sentiment model"
    }
]

model_comparison_df = spark.createDataFrame(comparison_data)

display(model_comparison_df)

correlation_with_rating,correlation_with_target,model,notes,sample_size
0.5597,0.5443,VADER,Lexicon-based baseline model,50000
0.6439,0.6277,DistilBERT,Transformer-based sentiment model,1000


In [0]:
# ---------------------------------------------------------
# Step 25: Save model comparison
# ---------------------------------------------------------
# Save model comparison results for reporting.

model_comparison_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_model_comparison"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_model_comparison")

Saved table: fashion_cip.silver.amazon_model_comparison


In [0]:
# ---------------------------------------------------------
# Step 26: Document model decision
# ---------------------------------------------------------
# Record the sentiment model selection decision.

decision_data = [
    {
        "selected_model": "DistilBERT",
        "baseline_model": "VADER",
        "decision_reason": "DistilBERT showed stronger alignment with review ratings and target labels in the test sample.",
        "fine_tuning_decision": "Pre-trained DistilBERT is used at this stage due to strong baseline performance and project time constraints.",
        "next_step": "Apply DistilBERT to categorized and aspect-tagged reviews once Cathrine's table is available."
    }
]

model_decision_df = spark.createDataFrame(decision_data)

display(model_decision_df)

baseline_model,decision_reason,fine_tuning_decision,next_step,selected_model
VADER,DistilBERT showed stronger alignment with review ratings and target labels in the test sample.,Pre-trained DistilBERT is used at this stage due to strong baseline performance and project time constraints.,Apply DistilBERT to categorized and aspect-tagged reviews once Cathrine's table is available.,DistilBERT


In [0]:
# ---------------------------------------------------------
# Step 27: Save model decision
# ---------------------------------------------------------
# Save model selection decision for documentation.

model_decision_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_model_decision"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_model_decision")

Saved table: fashion_cip.silver.amazon_model_decision


## Model Selection and Fine-Tuning Decision

VADER was used as a lexicon-based baseline model, while DistilBERT was tested as a transformer-based sentiment model. DistilBERT showed stronger alignment with the review ratings and target labels, so it was selected for the final sentiment scoring pipeline.

The final implementation uses the pre-trained `distilbert-base-uncased-finetuned-sst-2-english` model. Fine-tuning was not performed because the project did not include labeled aspect-level sentiment data for supervised training. Therefore, the pre-trained model was used as a practical and reproducible approach within the project scope. Future work could improve the model by fine-tuning DistilBERT on fashion-specific, aspect-level sentiment labels.


# Final Aspect-Level Sentiment Pipeline

The cells above are used for baseline exploration, including VADER scoring, DistilBERT testing, and model comparison.

The cells below form the final deliverable pipeline. This section uses the updated aspect-tagged NLP output, scores aspect-specific text spans using DistilBERT, aggregates sentiment by category and aspect, and writes the final Gold table for recommender integration.


In [0]:
# ---------------------------------------------------------
# Step 28: Check aspect-tagged review table
# ---------------------------------------------------------
# Verify that the aspect-tagged review table is available.

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"

aspect_table = f"{SILVER_SCHEMA}.amazon_aspect_tagged"

try:
    aspect_df = spark.table(aspect_table)
    
    print("Aspect-tagged table found:", aspect_table)
    print("Row count:", aspect_df.count())
    print("Column count:", len(aspect_df.columns))
    
    display(aspect_df.limit(10))
    aspect_df.printSchema()

except Exception as e:
    print("Aspect-tagged table is not available.")
    print("Expected table:", aspect_table)
    print("Error:", e)

Aspect-tagged table found: fashion_cip.silver.amazon_aspect_tagged
Row count: 927
Column count: 8


review_text,rating_clean,target_clean,predicted_category,category_confidence,aspect,span,matched_keyword
"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",4.0,1,Garment Lower body,0.27224260568618774,fit,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller).",fit
"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",4.0,1,Garment Lower body,0.27224260568618774,quality,Quality was cheap in some aspects but passable overall!,quality
"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",4.0,1,Garment Lower body,0.27224260568618774,value,Quality was cheap in some aspects but passable overall!,cheap
"Cute, a little loose but material feels good. Would order again!",4.0,1,Garment Lower body,0.18185661733150482,fit,"Cute, a little loose but material feels good.",loose
"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.",4.0,1,Garment Lower body,0.26647767424583435,fit,"I took off a star because it fits a little big, had to do an exchange.",fits
"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.",4.0,1,Garment Lower body,0.26647767424583435,comfort,Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dres

root
 |-- review_text: string (nullable = true)
 |-- rating_clean: double (nullable = true)
 |-- target_clean: long (nullable = true)
 |-- predicted_category: string (nullable = true)
 |-- category_confidence: double (nullable = true)
 |-- aspect: string (nullable = true)
 |-- span: string (nullable = true)
 |-- matched_keyword: string (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 29: Validate aspect table structure
# ---------------------------------------------------------
# Check required columns for sentiment aggregation.

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

aspect_table = f"{SILVER_SCHEMA}.amazon_aspect_tagged"
aspect_df = spark.table(aspect_table)

print("Table:", aspect_table)
print("Row count:", aspect_df.count())
print("Columns:")

for column in aspect_df.columns:
    print(column)

display(aspect_df.limit(10))

Table: fashion_cip.silver.amazon_aspect_tagged
Row count: 927
Columns:
review_text
rating_clean
target_clean
predicted_category
category_confidence
aspect
span
matched_keyword


review_text,rating_clean,target_clean,predicted_category,category_confidence,aspect,span,matched_keyword
"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",4.0,1,Garment Lower body,0.27224260568618774,fit,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller).",fit
"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",4.0,1,Garment Lower body,0.27224260568618774,quality,Quality was cheap in some aspects but passable overall!,quality
"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",4.0,1,Garment Lower body,0.27224260568618774,value,Quality was cheap in some aspects but passable overall!,cheap
"Cute, a little loose but material feels good. Would order again!",4.0,1,Garment Lower body,0.18185661733150482,fit,"Cute, a little loose but material feels good.",loose
"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.",4.0,1,Garment Lower body,0.26647767424583435,fit,"I took off a star because it fits a little big, had to do an exchange.",fits
"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.",4.0,1,Garment Lower body,0.26647767424583435,comfort,Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dres

In [0]:
# ---------------------------------------------------------
# Step 30: Prepare sentiment input
# ---------------------------------------------------------
# Use aspect-specific spans for final sentiment scoring.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

aspect_table = f"{SILVER_SCHEMA}.amazon_aspect_tagged"
aspect_df = spark.table(aspect_table)

sentiment_input_df = (
    aspect_df
    .withColumn("review_text", F.trim(F.col("review_text").cast("string")))
    .withColumn("sentiment_text", F.trim(F.col("span").cast("string")))
    .withColumn("rating", F.expr("try_cast(rating_clean as double)"))
    .withColumn("target", F.expr("try_cast(target_clean as int)"))
    .withColumn("category", F.trim(F.col("predicted_category").cast("string")))
    .withColumn("category_confidence", F.expr("try_cast(category_confidence as double)"))
    .withColumn("aspect", F.lower(F.trim(F.col("aspect").cast("string"))))
    .withColumn("matched_keyword", F.lower(F.trim(F.col("matched_keyword").cast("string"))))
    .withColumn(
        "review_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("review_text"), F.lit("")),
                F.coalesce(F.col("sentiment_text"), F.lit("")),
                F.coalesce(F.col("category"), F.lit("")),
                F.coalesce(F.col("aspect"), F.lit(""))
            ),
            256
        )
    )
    .filter(F.col("sentiment_text").isNotNull())
    .filter(F.length(F.col("sentiment_text")) > 0)
    .filter(F.col("category").isNotNull())
    .filter(F.col("aspect").isNotNull())
)

sentiment_input_df = sentiment_input_df.select(
    "review_id",
    "review_text",
    "sentiment_text",
    "rating",
    "target",
    "category",
    "category_confidence",
    "aspect",
    "matched_keyword"
)

sentiment_input_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_sentiment_input"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_sentiment_input")
print("Row count:", sentiment_input_df.count())

display(sentiment_input_df.limit(10))
sentiment_input_df.printSchema()

Saved table: fashion_cip.silver.amazon_sentiment_input
Row count: 927


review_id,review_text,sentiment_text,rating,target,category,category_confidence,aspect,matched_keyword
c717e188b20e4401c8be07c9b836bc6df6299b1308ad2851586c73b4f8996823,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!","I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller).",4.0,1,Garment Lower body,0.27224260568618774,fit,fit
7a11ff46a786fb31710e4534cb5f67a28c118c5144e065e32b578ba0b7d167cf,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",Quality was cheap in some aspects but passable overall!,4.0,1,Garment Lower body,0.27224260568618774,quality,quality
b7db0a86511e7d7ff7de07c5dbe9a4788611722a9114f3d775eacfde2c738b70,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",Quality was cheap in some aspects but passable overall!,4.0,1,Garment Lower body,0.27224260568618774,value,cheap
0a2f7b8553c0892752e9c55e46bcffe85e4622fe41c885e076f212ee6dd3e986,"Cute, a little loose but material feels good. Would order again!","Cute, a little loose but material feels good.",4.0,1,Garment Lower body,0.18185661733150482,fit,loose
5241a6f8f6a8ca3dc49bae78a3db817773687e5a1756ce4587ac63794fc3701d,"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.","I took off a star because it fits a little big, had to do an exchange.",4.0,1,Garment Lower body,0.26647767424583435,fit,fits
f3c43f34f5914dc4b3ffed9d5d9e050195bac0ebca3da917c93b9b60fd533c4c,"Not only is it beautiful (pictures donÃ¢ÂÂt do i

root
 |-- review_id: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- sentiment_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- target: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- category_confidence: double (nullable = true)
 |-- aspect: string (nullable = true)
 |-- matched_keyword: string (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 31: Save sentiment input
# ---------------------------------------------------------
# Save standardized aspect-tagged reviews.

sentiment_input_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_sentiment_input"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_sentiment_input")
print("Saved row count:", sentiment_input_df.count())

Saved table: fashion_cip.silver.amazon_sentiment_input
Saved row count: 927


In [0]:
# ---------------------------------------------------------
# Step 32A: Install transformer libraries
# ---------------------------------------------------------
# Install required libraries for DistilBERT sentiment scoring.

%pip install transformers torch -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ---------------------------------------------------------
# Step 32B: Restart Python
# ---------------------------------------------------------
# Restart Python so the installed packages are available.

dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------
# Step 32: Score aspect reviews with DistilBERT
# ---------------------------------------------------------
# Apply DistilBERT sentiment scoring to aspect-specific spans.

from transformers import pipeline
import pandas as pd
import numpy as np
import time

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"

sentiment_input_df = spark.table(f"{SILVER_SCHEMA}.amazon_sentiment_input")

sentiment_input_pdf = sentiment_input_df.toPandas()

print("Input rows:", len(sentiment_input_pdf))

distilbert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True
)

texts = sentiment_input_pdf["sentiment_text"].astype(str).tolist()

start_time = time.time()

distilbert_outputs = distilbert_model(
    texts,
    batch_size=32,
    truncation=True,
    max_length=512
)

end_time = time.time()

sentiment_input_pdf["sentiment_label"] = [item["label"] for item in distilbert_outputs]
sentiment_input_pdf["sentiment_score"] = [item["score"] for item in distilbert_outputs]

sentiment_input_pdf["sentiment_signed_score"] = np.where(
    sentiment_input_pdf["sentiment_label"] == "POSITIVE",
    sentiment_input_pdf["sentiment_score"],
    -sentiment_input_pdf["sentiment_score"]
)

runtime_seconds = end_time - start_time

print("DistilBERT scoring completed")
print("Runtime seconds:", round(runtime_seconds, 2))
print("Average seconds per record:", round(runtime_seconds / len(sentiment_input_pdf), 4))

display(spark.createDataFrame(sentiment_input_pdf).limit(10))

Input rows: 927


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT scoring completed
Runtime seconds: 27.65
Average seconds per record: 0.0298


review_id,review_text,sentiment_text,rating,target,category,category_confidence,aspect,matched_keyword,sentiment_label,sentiment_score,sentiment_signed_score
c717e188b20e4401c8be07c9b836bc6df6299b1308ad2851586c73b4f8996823,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!","I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller).",4.0,1,Garment Lower body,0.27224260568618774,fit,fit,POSITIVE,0.9224942326545715,0.9224942326545715
7a11ff46a786fb31710e4534cb5f67a28c118c5144e065e32b578ba0b7d167cf,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",Quality was cheap in some aspects but passable overall!,4.0,1,Garment Lower body,0.27224260568618774,quality,quality,POSITIVE,0.8163228034973145,0.8163228034973145
b7db0a86511e7d7ff7de07c5dbe9a4788611722a9114f3d775eacfde2c738b70,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",Quality was cheap in some aspects but passable overall!,4.0,1,Garment Lower body,0.27224260568618774,value,cheap,POSITIVE,0.8163228034973145,0.8163228034973145
0a2f7b8553c0892752e9c55e46bcffe85e4622fe41c885e076f212ee6dd3e986,"Cute, a little loose but material feels good. Would order again!","Cute, a little loose but material feels good.",4.0,1,Garment Lower body,0.18185661733150482,fit,loose,POSITIVE,0.9998248219490051,0.9998248219490051
5241a6f8f6a8ca3dc49bae78a3db817773687e5a1756ce4587ac63794fc3701d,"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.",

In [0]:
# ---------------------------------------------------------
# Step 33: Save DistilBERT aspect results
# ---------------------------------------------------------
# Save updated span-level sentiment results.

distilbert_aspect_results_df = spark.createDataFrame(sentiment_input_pdf)

distilbert_aspect_results_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.amazon_distilbert_aspect_results"
)

print("Saved table:", f"{SILVER_SCHEMA}.amazon_distilbert_aspect_results")
print("Saved row count:", distilbert_aspect_results_df.count())

display(distilbert_aspect_results_df.limit(10))

Saved table: fashion_cip.silver.amazon_distilbert_aspect_results
Saved row count: 927


review_id,review_text,sentiment_text,rating,target,category,category_confidence,aspect,matched_keyword,sentiment_label,sentiment_score,sentiment_signed_score
c717e188b20e4401c8be07c9b836bc6df6299b1308ad2851586c73b4f8996823,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!","I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller).",4.0,1,Garment Lower body,0.27224260568618774,fit,fit,POSITIVE,0.9224942326545715,0.9224942326545715
7a11ff46a786fb31710e4534cb5f67a28c118c5144e065e32b578ba0b7d167cf,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",Quality was cheap in some aspects but passable overall!,4.0,1,Garment Lower body,0.27224260568618774,quality,quality,POSITIVE,0.8163228034973145,0.8163228034973145
b7db0a86511e7d7ff7de07c5dbe9a4788611722a9114f3d775eacfde2c738b70,"I am 33-25-34 5'8'' and got a medium, which fit the way I wanted it too (I wanted it to be bigger so I looked smaller). However, the product photo, the other review photos, and what I received are all very different. Quality was cheap in some aspects but passable overall! Especially as a budget costume, not bad at all!The cap feather is very tiny and super bent. The bloomers / skirt are built into one, not like the product photo where they are separate. The socks will always fall down. The hat comes with no way to keep it on your head, so use pins.OH THE GLOVES were horrid. Thumb and pinkies on both gloves were twice the length they needed to be and looked like crazy alien fingers! What were they thinking lol.Some details seem to be off from other people's pics / product photos. But I stand by that this is a great budget Klee!",Quality was cheap in some aspects but passable overall!,4.0,1,Garment Lower body,0.27224260568618774,value,cheap,POSITIVE,0.8163228034973145,0.8163228034973145
0a2f7b8553c0892752e9c55e46bcffe85e4622fe41c885e076f212ee6dd3e986,"Cute, a little loose but material feels good. Would order again!","Cute, a little loose but material feels good.",4.0,1,Garment Lower body,0.18185661733150482,fit,loose,POSITIVE,0.9998248219490051,0.9998248219490051
5241a6f8f6a8ca3dc49bae78a3db817773687e5a1756ce4587ac63794fc3701d,"Not only is it beautiful (pictures donÃ¢ÂÂt do it justice) abd I receive so many compliments; itÃ¢ÂÂs super comfortable and definitely my favorite dress. I took off a star because it fits a little big, had to do an exchange.",

In [0]:
# ---------------------------------------------------------
# Step 34: Aggregate category-aspect sentiment
# ---------------------------------------------------------
# Aggregate span-level sentiment by category and aspect.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

distilbert_results_df = spark.table(f"{SILVER_SCHEMA}.amazon_distilbert_aspect_results")

category_aspect_sentiment_df = (
    distilbert_results_df
    .groupBy("category", "aspect")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("sentiment_signed_score"), 4).alias("mean_sentiment"),
        F.round(F.variance("sentiment_signed_score"), 4).alias("sentiment_variance"),
        F.round(F.avg("rating"), 2).alias("average_rating"),
        F.round(F.avg("category_confidence"), 4).alias("average_category_confidence"),
        F.round(
            F.avg(F.when(F.col("sentiment_label") == "POSITIVE", 1).otherwise(0)),
            4
        ).alias("positive_share"),
        F.round(
            F.avg(F.when(F.col("sentiment_label") == "NEGATIVE", 1).otherwise(0)),
            4
        ).alias("negative_share")
    )
    .filter(F.col("review_count") >= 5)
    .withColumn(
        "confidence",
        F.round(
            F.col("review_count") / (F.coalesce(F.col("sentiment_variance"), F.lit(0.0)) + F.lit(0.0001)),
            4
        )
    )
    .orderBy("category", "aspect")
)

print("Filtered category-aspect sentiment summary created")
print("Row count:", category_aspect_sentiment_df.count())

display(category_aspect_sentiment_df)

Filtered category-aspect sentiment summary created
Row count: 22


category,aspect,review_count,mean_sentiment,sentiment_variance,average_rating,average_category_confidence,positive_share,negative_share,confidence
Accessories,comfort,28,-0.2104,0.9302,3.0,0.3224,0.3929,0.6071,30.0978
Accessories,fit,56,-0.173,0.9543,3.16,0.3253,0.4107,0.5893,58.6756
Accessories,quality,32,-0.4917,0.7465,2.38,0.3328,0.25,0.75,42.861
Accessories,shipping,17,-0.493,0.7177,2.47,0.3326,0.2353,0.7647,23.6835
Accessories,value,27,-0.1219,0.9928,2.85,0.3396,0.4444,0.5556,27.1931
Bags,fit,12,-0.4897,0.7997,2.5,0.3802,0.25,0.75,15.0038
Bags,quality,9,-0.519,0.7445,2.33,0.4636,0.2222,0.7778,12.087
Bags,value,8,-0.2508,1.0617,2.88,0.5515,0.375,0.625,7.5344
Garment Full body,comfort,5,0.6274,0.6932,4.8,0.2056,0.8,0.2,7.2119
Garment Full body,fit,9,0.9998,0.0,4.78,0.18,1.0,0.0,90000.0


In [0]:
# ---------------------------------------------------------
# Step 35: Save category-aspect sentiment
# ---------------------------------------------------------
# Save final filtered sentiment features for recommender integration.

category_aspect_sentiment_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{GOLD_SCHEMA}.category_aspect_sentiment"
)

print("Saved table:", f"{GOLD_SCHEMA}.category_aspect_sentiment")
print("Saved row count:", category_aspect_sentiment_df.count())

Saved table: fashion_cip.gold.category_aspect_sentiment
Saved row count: 22


In [0]:
# ---------------------------------------------------------
# Step 36: Verify corrected Gold table
# ---------------------------------------------------------
# Confirm the final sentiment table after corrections.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"
GOLD_SCHEMA = f"{CATALOG}.gold"

gold_sentiment_df = spark.table(f"{GOLD_SCHEMA}.category_aspect_sentiment")

print("Gold table:", f"{GOLD_SCHEMA}.category_aspect_sentiment")
print("Row count:", gold_sentiment_df.count())
print("Column count:", len(gold_sentiment_df.columns))

display(gold_sentiment_df)

gold_sentiment_df.printSchema()

Gold table: fashion_cip.gold.category_aspect_sentiment
Row count: 22
Column count: 10


category,aspect,review_count,mean_sentiment,sentiment_variance,average_rating,average_category_confidence,positive_share,negative_share,confidence
Accessories,comfort,28,-0.2104,0.9302,3.0,0.3224,0.3929,0.6071,30.0978
Accessories,fit,56,-0.173,0.9543,3.16,0.3253,0.4107,0.5893,58.6756
Accessories,quality,32,-0.4917,0.7465,2.38,0.3328,0.25,0.75,42.861
Accessories,shipping,17,-0.493,0.7177,2.47,0.3326,0.2353,0.7647,23.6835
Accessories,value,27,-0.1219,0.9928,2.85,0.3396,0.4444,0.5556,27.1931
Bags,fit,12,-0.4897,0.7997,2.5,0.3802,0.25,0.75,15.0038
Bags,quality,9,-0.519,0.7445,2.33,0.4636,0.2222,0.7778,12.087
Bags,value,8,-0.2508,1.0617,2.88,0.5515,0.375,0.625,7.5344
Garment Full body,comfort,5,0.6274,0.6932,4.8,0.2056,0.8,0.2,7.2119
Garment Full body,fit,9,0.9998,0.0,4.78,0.18,1.0,0.0,90000.0


root
 |-- category: string (nullable = true)
 |-- aspect: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- mean_sentiment: double (nullable = true)
 |-- sentiment_variance: double (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- average_category_confidence: double (nullable = true)
 |-- positive_share: double (nullable = true)
 |-- negative_share: double (nullable = true)
 |-- confidence: double (nullable = true)



In [0]:
# ---------------------------------------------------------
# Step 37: Validate corrected Gold table
# ---------------------------------------------------------
# Check completeness, sentiment range, and minimum review count.

quality_check_df = gold_sentiment_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("category").isNull(), 1).otherwise(0)).alias("missing_category"),
    F.sum(F.when(F.col("aspect").isNull(), 1).otherwise(0)).alias("missing_aspect"),
    F.sum(F.when(F.col("mean_sentiment").isNull(), 1).otherwise(0)).alias("missing_mean_sentiment"),
    F.min("mean_sentiment").alias("min_mean_sentiment"),
    F.max("mean_sentiment").alias("max_mean_sentiment"),
    F.min("review_count").alias("min_review_count"),
    F.max("review_count").alias("max_review_count"),
    F.sum("review_count").alias("total_reviews_aggregated")
)

display(quality_check_df)

total_rows,missing_category,missing_aspect,missing_mean_sentiment,min_mean_sentiment,max_mean_sentiment,min_review_count,max_review_count,total_reviews_aggregated
22,0,0,0,-0.519,0.9998,5,267,873


In [0]:
# ---------------------------------------------------------
# Step 38: Save corrected Gold table validation
# ---------------------------------------------------------
# Save updated quality check results for documentation.

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"

quality_check_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_aspect_sentiment_quality_check"
)

print("Saved table:", f"{SILVER_SCHEMA}.category_aspect_sentiment_quality_check")

Saved table: fashion_cip.silver.category_aspect_sentiment_quality_check


In [0]:
# ---------------------------------------------------------
# Step 39: Log corrected final output to MLflow
# ---------------------------------------------------------
# Log final corrected sentiment output and validation metrics.

import mlflow
from pyspark.sql import functions as F

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

# Load corrected output tables.
aspect_results_df = spark.table(f"{SILVER_SCHEMA}.amazon_distilbert_aspect_results")
gold_sentiment_df = spark.table(f"{GOLD_SCHEMA}.category_aspect_sentiment")

# Calculate summary metrics.
input_record_count = aspect_results_df.count()
gold_row_count = gold_sentiment_df.count()

sentiment_metrics = gold_sentiment_df.select(
    F.min("mean_sentiment").alias("min_mean_sentiment"),
    F.max("mean_sentiment").alias("max_mean_sentiment"),
    F.avg("mean_sentiment").alias("avg_mean_sentiment"),
    F.min("review_count").alias("min_review_count"),
    F.max("review_count").alias("max_review_count"),
    F.sum("review_count").alias("total_reviews_aggregated")
).collect()[0]

# Set MLflow experiment.
mlflow.set_experiment("/Shared/fashion_cip_sentiment")

with mlflow.start_run(run_name="distilbert_span_category_aspect_sentiment"):
    mlflow.log_param("model", "distilbert-base-uncased-finetuned-sst-2-english")
    mlflow.log_param("sentiment_text_source", "span")
    mlflow.log_param("minimum_review_count_filter", 5)
    mlflow.log_param("input_table", f"{SILVER_SCHEMA}.amazon_sentiment_input")
    mlflow.log_param("output_table", f"{GOLD_SCHEMA}.category_aspect_sentiment")
    
    mlflow.log_metric("input_record_count", float(input_record_count))
    mlflow.log_metric("gold_row_count", float(gold_row_count))
    mlflow.log_metric("min_mean_sentiment", float(sentiment_metrics["min_mean_sentiment"]))
    mlflow.log_metric("max_mean_sentiment", float(sentiment_metrics["max_mean_sentiment"]))
    mlflow.log_metric("avg_mean_sentiment", float(sentiment_metrics["avg_mean_sentiment"]))
    mlflow.log_metric("min_review_count", float(sentiment_metrics["min_review_count"]))
    mlflow.log_metric("max_review_count", float(sentiment_metrics["max_review_count"]))
    mlflow.log_metric("total_reviews_aggregated", float(sentiment_metrics["total_reviews_aggregated"]))

print("MLflow logging completed")
print("Input record count:", input_record_count)
print("Gold row count:", gold_row_count)

MLflow logging completed
Input record count: 927
Gold row count: 22


In [0]:
# ---------------------------------------------------------
# Step 40: Update handoff schema summary
# ---------------------------------------------------------
# Document the corrected final sentiment table columns.

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"

handoff_schema_data = [
    {
        "column_name": "category",
        "description": "Predicted product category from the NLP classification step.",
        "usage": "Used as a grouping feature for recommender integration."
    },
    {
        "column_name": "aspect",
        "description": "Aspect extracted from review text, such as fit, comfort, quality, value, or shipping.",
        "usage": "Used to understand customer sentiment by product issue or strength."
    },
    {
        "column_name": "review_count",
        "description": "Number of aspect-level records in each category-aspect group.",
        "usage": "Groups with fewer than 5 reviews are excluded from the final Gold table."
    },
    {
        "column_name": "mean_sentiment",
        "description": "Average signed DistilBERT sentiment score based on aspect-specific spans.",
        "usage": "Main sentiment feature for recommender integration and business analysis."
    },
    {
        "column_name": "sentiment_variance",
        "description": "Variation in sentiment scores within each category-aspect group.",
        "usage": "Used to evaluate consistency of sentiment within the group."
    },
    {
        "column_name": "confidence",
        "description": "Confidence score adjusted using review count and sentiment variance.",
        "usage": "Higher values indicate stronger and more consistent sentiment evidence."
    },
    {
        "column_name": "average_rating",
        "description": "Average star rating within each category-aspect group.",
        "usage": "Used as a validation feature alongside sentiment."
    },
    {
        "column_name": "average_category_confidence",
        "description": "Average confidence score from the category classification step.",
        "usage": "Indicates reliability of the predicted category assignment."
    },
    {
        "column_name": "positive_share",
        "description": "Share of aspect-level records classified as positive.",
        "usage": "Business-friendly indicator of positive customer perception."
    },
    {
        "column_name": "negative_share",
        "description": "Share of aspect-level records classified as negative.",
        "usage": "Used to identify potential product concerns."
    }
]

handoff_schema_df = spark.createDataFrame(handoff_schema_data)

handoff_schema_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_aspect_sentiment_schema"
)

print("Saved table:", f"{SILVER_SCHEMA}.category_aspect_sentiment_schema")

display(handoff_schema_df)

Saved table: fashion_cip.silver.category_aspect_sentiment_schema


column_name,description,usage
category,Predicted product category from the NLP classification step.,Used as a grouping feature for recommender integration.
aspect,"Aspect extracted from review text, such as fit, comfort, quality, value, or shipping.",Used to understand customer sentiment by product issue or strength.
review_count,Number of aspect-level records in each category-aspect group.,Groups with fewer than 5 reviews are excluded from the final Gold table.
mean_sentiment,Average signed DistilBERT sentiment score based on aspect-specific spans.,Main sentiment feature for recommender integration and business analysis.
sentiment_variance,Variation in sentiment scores within each category-aspect group.,Used to evaluate consistency of sentiment within the group.
confidence,Confidence score adjusted using review count and sentiment variance.,Higher values indicate stronger and more consistent sentiment evidence.
average_rating,Average star rating within each category-aspect group.,Used as a validation feature alongside sentiment.
average_category_confidence,Average confidence score from the category classification step.,Indicates reliability of the predicted category assignment.
positive_share,Share of aspect-level records classified as positive.,Business-friendly indicator of positive customer perception.
negative_share,Share of aspect-level records classified as negative.,Used to identify potential product concerns.


In [0]:
# ---------------------------------------------------------
# Step 41: Save handoff schema summary
# ---------------------------------------------------------
# Save final table documentation for integration.

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"

handoff_schema_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_aspect_sentiment_schema"
)

print("Saved table:", f"{SILVER_SCHEMA}.category_aspect_sentiment_schema")

Saved table: fashion_cip.silver.category_aspect_sentiment_schema


In [0]:
# ---------------------------------------------------------
# Step 42: Update sentiment insight tables
# ---------------------------------------------------------
# Recreate top positive and negative summaries from corrected Gold table.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"
GOLD_SCHEMA = f"{CATALOG}.gold"
SILVER_SCHEMA = f"{CATALOG}.silver"

gold_sentiment_df = spark.table(f"{GOLD_SCHEMA}.category_aspect_sentiment")

top_positive_df = (
    gold_sentiment_df
    .orderBy(F.col("mean_sentiment").desc())
    .limit(10)
)

top_negative_df = (
    gold_sentiment_df
    .orderBy(F.col("mean_sentiment").asc())
    .limit(10)
)

top_positive_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_aspect_top_positive"
)

top_negative_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_aspect_top_negative"
)

print("Saved table:", f"{SILVER_SCHEMA}.category_aspect_top_positive")
print("Saved table:", f"{SILVER_SCHEMA}.category_aspect_top_negative")

print("Top positive sentiment:")
display(top_positive_df)

print("Top negative sentiment:")
display(top_negative_df)


Saved table: fashion_cip.silver.category_aspect_top_positive
Saved table: fashion_cip.silver.category_aspect_top_negative
Top positive sentiment:


category,aspect,review_count,mean_sentiment,sentiment_variance,average_rating,average_category_confidence,positive_share,negative_share,confidence
Garment Full body,fit,9,0.9998,0.0,4.78,0.18,1.0,0.0,90000.0
Garment Full body,value,5,0.9993,0.0,4.6,0.1778,1.0,0.0,50000.0
Garment Full body,quality,7,0.9367,0.0278,4.86,0.1853,1.0,0.0,250.8961
Garment Full body,comfort,5,0.6274,0.6932,4.8,0.2056,0.8,0.2,7.2119
Garment Lower body,comfort,108,-0.0147,0.9384,3.18,0.2374,0.5,0.5,115.0773
Garment Upper body,fit,20,-0.0856,1.0017,2.7,0.2148,0.45,0.55,19.9641
Swimwear,fit,13,-0.0894,1.0439,3.31,0.4142,0.4615,0.5385,12.4521
Accessories,value,27,-0.1219,0.9928,2.85,0.3396,0.4444,0.5556,27.1931
Garment Lower body,shipping,28,-0.1646,0.9544,3.0,0.2342,0.4286,0.5714,29.3347
Accessories,fit,56,-0.173,0.9543,3.16,0.3253,0.4107,0.5893,58.6756


Top negative sentiment:


category,aspect,review_count,mean_sentiment,sentiment_variance,average_rating,average_category_confidence,positive_share,negative_share,confidence
Bags,quality,9,-0.519,0.7445,2.33,0.4636,0.2222,0.7778,12.087
Accessories,shipping,17,-0.493,0.7177,2.47,0.3326,0.2353,0.7647,23.6835
Accessories,quality,32,-0.4917,0.7465,2.38,0.3328,0.25,0.75,42.861
Bags,fit,12,-0.4897,0.7997,2.5,0.3802,0.25,0.75,15.0038
Shoes,comfort,13,-0.3781,0.9139,3.31,0.5155,0.3077,0.6923,14.2232
Shoes,fit,17,-0.2965,0.9524,2.94,0.4264,0.3529,0.6471,17.8478
Garment Lower body,value,79,-0.2914,0.9114,2.61,0.2214,0.3544,0.6456,86.6703
Garment Lower body,quality,108,-0.2602,0.9118,2.79,0.2129,0.3704,0.6296,118.434
Bags,value,8,-0.2508,1.0617,2.88,0.5515,0.375,0.625,7.5344
Accessories,comfort,28,-0.2104,0.9302,3.0,0.3224,0.3929,0.6071,30.0978


In [0]:
# ---------------------------------------------------------
# Step 43: Update category sentiment summary
# ---------------------------------------------------------
# Recreate category-level summary from corrected Gold table.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"
GOLD_SCHEMA = f"{CATALOG}.gold"
SILVER_SCHEMA = f"{CATALOG}.silver"

gold_sentiment_df = spark.table(f"{GOLD_SCHEMA}.category_aspect_sentiment")

category_sentiment_summary_df = (
    gold_sentiment_df
    .groupBy("category")
    .agg(
        F.sum("review_count").alias("total_reviews"),
        F.round(F.avg("mean_sentiment"), 4).alias("avg_category_sentiment"),
        F.round(F.avg("average_rating"), 2).alias("avg_category_rating"),
        F.round(F.avg("positive_share"), 4).alias("avg_positive_share"),
        F.round(F.avg("negative_share"), 4).alias("avg_negative_share"),
        F.countDistinct("aspect").alias("aspect_count")
    )
    .orderBy(F.col("avg_category_sentiment").desc())
)

category_sentiment_summary_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_sentiment_summary"
)

print("Saved table:", f"{SILVER_SCHEMA}.category_sentiment_summary")
print("Saved row count:", category_sentiment_summary_df.count())

display(category_sentiment_summary_df)

Saved table: fashion_cip.silver.category_sentiment_summary
Saved row count: 7


category,total_reviews,avg_category_sentiment,avg_category_rating,avg_positive_share,avg_negative_share,aspect_count
Garment Full body,26,0.8908,4.76,0.95,0.05,4
Garment Upper body,20,-0.0856,2.7,0.45,0.55,1
Swimwear,13,-0.0894,3.31,0.4615,0.5385,1
Garment Lower body,590,-0.1872,2.89,0.4101,0.5899,5
Shoes,35,-0.2915,3.08,0.3535,0.6465,3
Accessories,160,-0.298,2.77,0.3467,0.6533,5
Bags,29,-0.4198,2.57,0.2824,0.7176,3


In [0]:
# ---------------------------------------------------------
# Step 44: Update limitation note
# ---------------------------------------------------------
# Document final limitations after pipeline correction.

from pyspark.sql import functions as F

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

sentiment_input_df = spark.table(f"{SILVER_SCHEMA}.amazon_sentiment_input")
gold_sentiment_df = spark.table(f"{GOLD_SCHEMA}.category_aspect_sentiment")

input_row_count = sentiment_input_df.count()
gold_row_count = gold_sentiment_df.count()

limitation_data = [
    {
        "area": "Aspect-level sentiment analysis",
        "limitation": f"The final sentiment pipeline uses {input_row_count} aspect-tagged span records and produces {gold_row_count} category-aspect groups after applying a minimum review count filter.",
        "impact": "The pipeline now scores aspect-specific spans instead of full reviews, but results are still dependent on the size and quality of the aspect-tagged NLP input.",
        "recommendation": "Scale the NLP categorization and aspect extraction process to a larger review sample or full dataset for stronger business conclusions."
    },
    {
        "area": "Model training",
        "limitation": "A pre-trained DistilBERT SST-2 model was used instead of fine-tuning.",
        "impact": "The model provides strong baseline sentiment performance, but it is not specifically fine-tuned for fashion product aspect sentiment.",
        "recommendation": "Fine-tune the model in future work if labeled aspect-level sentiment data becomes available."
    }
]

limitation_df = spark.createDataFrame(limitation_data)

limitation_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.category_aspect_sentiment_limitations"
)

print("Saved table:", f"{SILVER_SCHEMA}.category_aspect_sentiment_limitations")

display(limitation_df)

Saved table: fashion_cip.silver.category_aspect_sentiment_limitations


area,impact,limitation,recommendation
Aspect-level sentiment analysis,"The pipeline now scores aspect-specific spans instead of full reviews, but results are still dependent on the size and quality of the aspect-tagged NLP input.",The final sentiment pipeline uses 927 aspect-tagged span records and produces 22 category-aspect groups after applying a minimum review count filter.,Scale the NLP categorization and aspect extraction process to a larger review sample or full dataset for stronger business conclusions.
Model training,"The model provides strong baseline sentiment performance, but it is not specifically fine-tuned for fashion product aspect sentiment.",A pre-trained DistilBERT SST-2 model was used instead of fine-tuning.,Fine-tune the model in future work if labeled aspect-level sentiment data becomes available.


In [0]:
# ---------------------------------------------------------
# Step 45: Create final output checklist
# ---------------------------------------------------------
# Verify that all required sentiment output tables exist.

CATALOG = "fashion_cip"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

required_tables = [
    f"{SILVER_SCHEMA}.amazon_sentiment_input",
    f"{SILVER_SCHEMA}.amazon_distilbert_aspect_results",
    f"{GOLD_SCHEMA}.category_aspect_sentiment",
    f"{SILVER_SCHEMA}.category_aspect_sentiment_quality_check",
    f"{SILVER_SCHEMA}.category_aspect_sentiment_schema",
    f"{SILVER_SCHEMA}.category_aspect_top_positive",
    f"{SILVER_SCHEMA}.category_aspect_top_negative",
    f"{SILVER_SCHEMA}.category_sentiment_summary",
    f"{SILVER_SCHEMA}.category_aspect_sentiment_limitations"
]

checklist = []

for table_name in required_tables:
    try:
        df = spark.table(table_name)
        checklist.append({
            "table_name": table_name,
            "status": "Available",
            "row_count": df.count()
        })
    except Exception:
        checklist.append({
            "table_name": table_name,
            "status": "Missing",
            "row_count": None
        })

final_checklist_df = spark.createDataFrame(checklist)

final_checklist_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER_SCHEMA}.sentiment_output_checklist"
)

print("Saved table:", f"{SILVER_SCHEMA}.sentiment_output_checklist")

display(final_checklist_df)
print("Saved table:", f"{SILVER_SCHEMA}.sentiment_output_checklist")

Saved table: fashion_cip.silver.sentiment_output_checklist


row_count,status,table_name
927,Available,fashion_cip.silver.amazon_sentiment_input
927,Available,fashion_cip.silver.amazon_distilbert_aspect_results
22,Available,fashion_cip.gold.category_aspect_sentiment
1,Available,fashion_cip.silver.category_aspect_sentiment_quality_check
10,Available,fashion_cip.silver.category_aspect_sentiment_schema
10,Available,fashion_cip.silver.category_aspect_top_positive
10,Available,fashion_cip.silver.category_aspect_top_negative
7,Available,fashion_cip.silver.category_sentiment_summary
2,Available,fashion_cip.silver.category_aspect_sentiment_limitations


Saved table: fashion_cip.silver.sentiment_output_checklist
